First import modules and initialize Earth Engine.

In [2]:

# standard modules
import io
import json
import os
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project='nr218-michaelhuggins')

Open the AOI vector file and check it on a map.

In [4]:
# read AOIs
# This works whether the notebook is launched from the repo root or from assets/.
this_dir = Path.cwd()
repo_dir = this_dir if (this_dir / 'assets').exists() else this_dir.parent

def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

# Broad analysis/export AOI.
aoi_path = first_existing_path([
    this_dir / 'rio_san_juan_aoi.geojson',
    repo_dir / 'assets' / 'rio_san_juan_aoi.geojson',
])

# Smaller comparison/reference AOI inside the Rio San Juan box.
rio_aoi_path = first_existing_path([
    this_dir / 'rio_indio_aoi.geojson',
    repo_dir / 'untracked_qgis' / 'susques' / 'rio_indio_aoi.geojson',
])

aoi_path, rio_aoi_path

(PosixPath('/home/michael/Work/nr218/assets/rio_san_juan_aoi.geojson'),
 PosixPath('/home/michael/Work/nr218/assets/rio_indio_aoi.geojson'))

In [5]:
aoi = gpd.read_file(aoi_path).to_crs(4326)
rio_aoi = gpd.read_file(rio_aoi_path).to_crs(4326)

gee_json = json.loads(aoi[['geometry']].to_json())
gee_aoi = geemap.geojson_to_ee(gee_json)

gee_rio_json = json.loads(rio_aoi[['geometry']].to_json())
gee_rio_aoi = geemap.geojson_to_ee(gee_rio_json)

# inspect the GeoJSON as an EEObject through geemap.
test_map = geemap.Map(basemap='SATELLITE')
test_map.centerObject(gee_aoi, 9)

# Style the AOIs explicitly so both are visible on the map.
aoi_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
rio_aoi_style = {'color': 'FFFF00', 'fillColor': '00000000', 'width': 3}
test_map.addLayer(gee_aoi.style(**aoi_style), {}, 'Broad Rio San Juan AOI')
test_map.addLayer(gee_rio_aoi.style(**rio_aoi_style), {}, 'Rio Indio AOI')

test_map


Map(center=[10.990011493745822, -83.83500000000112], controls=(WidgetControl(options=['position', 'transparent…

Get the vertices of the broad AOI and Rio Indio AOI to use in Earth Engine filters.


In [6]:
# get extents for both AOIs
def bounds_to_verts(gdf):
    minx, miny, maxx, maxy = gdf.total_bounds
    return [
        [float(minx), float(miny)],
        [float(minx), float(maxy)],
        [float(maxx), float(maxy)],
        [float(maxx), float(miny)],
        [float(minx), float(miny)],
    ]

# Broad box from the original Rio San Juan AOI.
verts = bounds_to_verts(aoi)

# Smaller AOI used for Rio Indio wet-season composites.
rio_verts = bounds_to_verts(rio_aoi)

verts, rio_verts


([[-84.05, 10.88],
  [-84.05, 11.1],
  [-83.62, 11.1],
  [-83.62, 10.88],
  [-84.05, 10.88]],
 [[-83.79898991031392, 10.900770179372204],
  [-83.79898991031392, 11.025720852017939],
  [-83.62053026905832, 11.025720852017939],
  [-83.62053026905832, 10.900770179372204],
  [-83.79898991031392, 10.900770179372204]])

Wet-season composite periods

Use these periods to download annual wet-season composites for the Rio Indio AOI. 


In [9]:
# wet season for the Caribbean side of Nicaragua.
# end dates are exclusive, so November composites end on December 1.
WET_START_MONTH = 5
WET_END_MONTH = 11

composite_periods = {
    'landsat_5_7': {
        'start_year': 2000,
        'end_year': 2011,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l57',
    },
    'landsat_8_9': {
        'start_year': 2013,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l89',
    },
    'sentinel_2': {
        'start_year': 2018,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 10,
        'short_name': 's2',
    },
}

# Use this region for Rio Indio downloads.
rio_export_region = gee_rio_aoi.geometry()


def wet_season_date_range(year):
    start = ee.Date.fromYMD(year, WET_START_MONTH, 1)
    end = ee.Date.fromYMD(year, WET_END_MONTH, 1).advance(1, 'month')
    return start, end


# Quick check: these are the image years that will be requested for each sensor group.
for sensor, period in composite_periods.items():
    years = list(range(period['start_year'], period['end_year'] + 1))
    print(sensor, years[0], 'to', years[-1], f"({len(years)} composites)")


landsat_5_7 2000 to 2011 (12 composites)
landsat_8_9 2013 to 2025 (13 composites)
sentinel_2 2018 to 2025 (8 composites)


Export annual composites to Google Drive.

Run the helper cell, then call `queue_drive_exports('sentinel_2')` or another sensor key from `composite_periods`.


In [10]:
COMMON_BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
DRIVE_FOLDER = 'nicaragua_wet_season'


def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = 1 << 4
    clouds = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(qa.bitwiseAnd(clouds).eq(0))
    return image.updateMask(mask)


def prep_landsat_5_7(image):
    optical = image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def prep_landsat_8_9(image):
    optical = image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def mask_s2_clouds(image):
    qa = image.select('QA60')
    clouds = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(clouds).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    optical = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'], COMMON_BANDS)
    return optical.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())


def build_wet_season_composite(sensor, year, region):
    period = composite_periods[sensor]
    start, end = wet_season_date_range(year)

    if sensor == 'landsat_5_7':
        collection = (
            ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .map(mask_landsat_sr)
            .map(prep_landsat_5_7)
        )
    elif sensor == 'landsat_8_9':
        collection = (
            ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .map(mask_landsat_sr)
            .map(prep_landsat_8_9)
        )
    elif sensor == 'sentinel_2':
        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35))
            .map(mask_s2_clouds)
        )
    else:
        raise ValueError(f'Unknown sensor: {sensor}')

    return collection.median().clip(region)


def queue_drive_exports(sensor='sentinel_2', folder=DRIVE_FOLDER, region=rio_export_region):
    period = composite_periods[sensor]
    tasks = []

    for year in range(period['start_year'], period['end_year'] + 1):
        image = build_wet_season_composite(sensor, year, region)
        prefix = f"{period['short_name']}_{year}_wet_season"
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=prefix,
            folder=folder,
            fileNamePrefix=prefix,
            region=region,
            scale=period['scale'],
            fileFormat='GeoTIFF',
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)
        print(f'Started export: {prefix}')

    return tasks


# Example:
# tasks = queue_drive_exports('sentinel_2')


In [ ]:
tasks = []

for sensor in composite_periods.keys():
    tasks.extend(queue_drive_exports(sensor))

print(f'Queued {len(tasks)} export tasks.')

In [ ]:
flat_tasks = []
for task in tasks:
    if isinstance(task, list):
        flat_tasks.extend(task)
    else:
        flat_tasks.append(task)

seen_done = set()
terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}
not_done = True

while not_done:
    for task in flat_tasks:
        status = task.status()
        description = status.get('description', task.id)
        state = status.get('state', 'UNKNOWN')

        if state in terminal_states and description not in seen_done:
            print(description, state)
            seen_done.add(description)

    not_done = any(task.status().get('state') not in terminal_states for task in flat_tasks)

    if not_done:
        print('Waiting 3 minutes before checking again...')
        time.sleep(180)

print('All tasks are done.')